# Band-edge filter design comparison

This notebook is the slower companion to `notes/band-edge-filter-shape-and-slope.md`.

The narrow question is now implementation-shaped: **how much of the remaining near-lock slope gap came from the current proxy filter shape rather than from the band-edge idea itself?**


## Local setup

- SRRC QPSK
- `4 samples/symbol`
- `1024` symbols
- central finite difference at `±0.01 R_s`
- tap counts `{63, 127, 255}`
- roll-off `{0.05, 0.20, 0.35, 0.50}`
- two designs: current proxy bandpass and GNU Radio / half-sine style


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

rows = []
with (Path('..') / 'assets' / '2026-05-22-band-edge-filter-design-comparison.csv').open() as handle:
    for row in csv.DictReader(handle):
        parsed = {}
        for key, value in row.items():
            if key in {'design'}:
                parsed[key] = value
            elif key in {'samples_per_symbol', 'symbol_count', 'seed', 'trim', 'tap_count'}:
                parsed[key] = int(value)
            else:
                parsed[key] = float(value)
        rows.append(parsed)

len(rows), rows[0].keys()


## One direct comparison

The clearest check is `α = 0.35` with 63 taps. That is exactly where the current proxy still looks soft while the GNU Radio / half-sine construction is already near the normalized target slope.


In [ ]:
for row in rows:
    if row['tap_count'] == 63 and row['rolloff'] == 0.35:
        print(row['design'], row['central_slope_wrt_deltaf_over_Rs'], row['imbalance_at_0p10'])


## Group by design and tap count

This is the compact table that makes the whole pass worth keeping.


In [ ]:
grouped = defaultdict(lambda: defaultdict(list))
for row in rows:
    grouped[row['design']][row['tap_count']].append(row)

for design, tap_map in grouped.items():
    print(f'\n{design}')
    for tap_count, series in sorted(tap_map.items()):
        print(f'  {tap_count} taps')
        for row in sorted(series, key=lambda entry: entry['rolloff']):
            print(
                f"    alpha={row['rolloff']:.2f} slope={row['central_slope_wrt_deltaf_over_Rs']:.3f} raw@0.10={row['imbalance_at_0p10']:.3f}"
            )


## Takeaway

- the current proxy still works as a first intuition panel
- the GNU Radio / half-sine shape reaches the normalized slope target much sooner
- `α = 0.05` remains the real weak case either way
- the next bounded question is guardband / adjacent-channel cost, not more slope sweeps
